# Late Interaction vs Dense Retrieval Demo
## Why ColBERT beats traditional RAG on complex queries

---

### 🎯 Demo Goal
Show why **late interaction** (ColBERT) outperforms **dense retrieval** using real restaurant reviews that everyone can understand.

### 📊 What we'll demonstrate:
1. **Multi-constraint queries** - "Italian + budget-friendly + outdoor seating"
2. **Contradictory concepts** - "Expensive but worth it" 
3. **Token-level visualization** - See WHY ColBERT succeeds

## 🔍 Dense vs Late Interaction - The Key Difference

### Dense Retrieval (Traditional RAG)
```
Query: "Italian budget-friendly outdoor"
         ↓
    [Single Vector]
         ↓  
    Document: "Amazing Italian restaurant..."
         ↓
    [Single Vector]
         ↓
    Cosine Similarity → Score
```
**Problem**: All information compressed into one point - relationships lost!

### Late Interaction (ColBERT)
```
Query: "Italian budget-friendly outdoor"
         ↓
    [Token₁] [Token₂] [Token₃]
         ↓     ↓      ↓
    Document: "Amazing Italian restaurant..."
         ↓
    [Token₁] [Token₂] ... [TokenN]
         ↓
    MaxSim(each query token vs all doc tokens) → Score
```
**Solution**: Every token preserved - relationships maintained!

## 📦 Setup - Install Required Packages

In [ ]:
# Import shared configuration using clean setup
from setup import *

print("✅ Notebook setup complete!")
print(f"📁 Using data from: {os.getenv('RESTAURANT_REVIEWS_CSV')}")
print(f"🗄️ Vector store path: {os.getenv('LANCEDB_PATH')}")
print(f"🤖 Dense model: {os.getenv('DENSE_MODEL_NAME')}")
print(f"🔍 ColBERT model: {os.getenv('COLBERT_MODEL_NAME')}")

In [3]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from pylate import models, evaluation
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Libraries loaded successfully!")
print(f"📱 PyTorch version: {torch.__version__}")
print(f"🍎 MPS (M1/M2) available: {torch.backends.mps.is_available()}")
print(f"🔥 CUDA available: {torch.cuda.is_available()}")

# Set device for M1 Mac optimization
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"🚀 Using M1/M2 GPU acceleration: {device}")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"🚀 Using CUDA GPU acceleration: {device}")
else:
    device = torch.device("cpu")
    print(f"💻 Using CPU: {device}")
    
# Set global device for all models
torch.set_default_device(device)

/Users/luvsuneja/Documents/repos/advanced-rag-experimentation/.venv/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


✅ Libraries loaded successfully!
📱 PyTorch version: 2.2.2
🍎 MPS (M1/M2) available: True
🔥 CUDA available: False
🚀 Using M1/M2 GPU acceleration: mps


In [ ]:
# Load the restaurant reviews data
df = pd.read_csv('../data/restaurant_reviews.csv')

print(f"✅ Loaded {len(df)} restaurant reviews")
print(f"📊 Columns: {df.columns.tolist()}")
print(f"⭐ Average rating: {df['rating'].mean():.1f}/5")
print("\n📝 First review preview:")
print(f"Restaurant: {df.iloc[0]['restaurant']}")
print(f"Review: {df.iloc[0]['review'][:150]}...")
print(f"Rating: {'⭐' * df.iloc[0]['rating']}")

# Display the dataframe
df.head()

In [ ]:
# Load the restaurant reviews data using .env configuration
df = pd.read_csv(os.getenv('RESTAURANT_REVIEWS_CSV'))

print(f"✅ Loaded {len(df)} restaurant reviews")
print(f"📊 Columns: {df.columns.tolist()}")
print(f"⭐ Average rating: {df['rating'].mean():.1f}/5")
print("\n📝 First review preview:")
print(f"Restaurant: {df.iloc[0]['restaurant']}")
print(f"Review: {df.iloc[0]['review'][:150]}...")
print(f"Rating: {'⭐' * df.iloc[0]['rating']}")

# Display the dataframe
df.head()

In [ ]:
import lancedb

# Create LanceDB connection (local database) - consistent path
db_path = "./restaurant_reviews_vectors"
db = lancedb.connect(db_path)

print(f"✅ Connected to LanceDB at: {db_path}")
print(f"📊 Tables in database: {db.table_names()}")

# We'll create two tables - one for dense, one for ColBERT
print("\n🎯 We'll create separate tables for:")
print("  1. dense_reviews - Traditional single vector per document")
print("  2. colbert_reviews - Multiple vectors per document")

In [ ]:
# Load Dense Retrieval Model (Traditional RAG)
from sentence_transformers import SentenceTransformer

print("🔄 Loading dense model...")

# Explicitly set device for M1
import torch
if torch.backends.mps.is_available():
    model_device = 'mps'
    print("🍎 Using M1 GPU (MPS) for dense model")
else:
    model_device = 'cpu'
    print("💻 Using CPU for dense model")

# Load model with explicit device setting
dense_model = SentenceTransformer(
    'all-MiniLM-L6-v2',
    device=model_device
)

print(f"✅ Dense model loaded: all-MiniLM-L6-v2")
print(f"📏 Dense embedding dimension: 384")
print(f"🎯 Creates: 1 vector per document")
print(f"🔧 Running on: {model_device}")

# Test dense embedding
test_text = "Italian restaurant with outdoor seating"
test_dense_embedding = dense_model.encode(test_text)
print(f"\n🧪 Test dense embedding:")
print(f"   Input: '{test_text}'")
print(f"   Output shape: {test_dense_embedding.shape} (single vector)")

In [ ]:
import lancedb

# Create LanceDB connection using .env configuration
db = lancedb.connect(os.getenv('LANCEDB_PATH'))

print(f"✅ Connected to LanceDB at: {os.getenv('LANCEDB_PATH')}")
print(f"📊 Tables in database: {db.table_names()}")

# We'll create two tables - one for dense, one for ColBERT
print("\n🎯 We'll create separate tables for:")
print("  1. dense_reviews - Traditional single vector per document")
print("  2. colbert_reviews - Multiple vectors per document")

In [ ]:
# Load Dense Retrieval Model using .env configuration
from sentence_transformers import SentenceTransformer
import torch

print("🔄 Loading dense model...")

# Use .env device configuration
model_device = get_device()
if model_device == 'mps':
    model_device = 'cpu'  # Sentence transformers work better on CPU for M1
    print("🍎 Using CPU for dense model (optimized for M1)")
else:
    print(f"💻 Using {model_device} for dense model")

# Load model using .env configuration
dense_model = SentenceTransformer(os.getenv('DENSE_MODEL_NAME'), device=model_device)

print(f"✅ Dense model loaded: {os.getenv('DENSE_MODEL_NAME')}")
print(f"📏 Dense embedding dimension: {os.getenv('EMBEDDING_DIMENSION', 384)}")
print(f"🎯 Creates: 1 vector per document")
print(f"🔧 Running on: {model_device}")

# Test dense embedding
test_text = "Italian restaurant with outdoor seating"
test_dense_embedding = dense_model.encode(test_text)
print(f"\n🧪 Test dense embedding:")
print(f"   Input: '{test_text}'")
print(f"   Output shape: {test_dense_embedding.shape} (single vector)")

In [ ]:
# Visualize the difference
print("🎨 VISUAL COMPARISON")
print("=" * 60)
print("\n📦 DENSE RETRIEVAL (Traditional RAG):")
print(f"   '{test_text}'")
print(f"   ↓")
print(f"   [Single 384-dim vector] → Compressed into 1 point")
print(f"   Shape: {test_dense_embedding.shape}")

print("\n🔍 COLBERT (Late Interaction):")
print(f"   '{test_text}'")
print(f"   ↓")
tokens = test_text.split()  # Simplified tokenization for display
print(f"   {tokens}")
print(f"   ↓")
print(f"   [Vector for 'Italian'] [Vector for 'restaurant'] [Vector for 'with'] ...")
print(f"   Shape: {test_colbert_embedding[0].shape}")
print(f"\n💡 Key Insight: ColBERT preserves token-level information!")
print("   This allows matching 'Italian' separately from 'outdoor seating'")

In [ ]:
# Load ColBERT Model using .env configuration
from pylate import models

print("🔄 Loading ColBERT model...")

# Use .env device configuration - ColBERT works better on CPU for M1
colbert_device = 'cpu'
print("🍎 Using CPU for ColBERT model (optimized for stability)")

# Initialize ColBERT using .env configuration
colbert_model = models.ColBERT(
    model_name_or_path=os.getenv('COLBERT_MODEL_NAME'),
    device=colbert_device
)

print(f"✅ ColBERT model loaded")
print(f"📏 ColBERT embedding dimension: {os.getenv('EMBEDDING_DIMENSION', 384)} per token")
print(f"🎯 Creates: Multiple vectors (1 per token) per document")
print(f"🔧 Running on: {colbert_device}")

# Test ColBERT embedding - note it returns multiple vectors
print(f"\n🧪 Test ColBERT embedding:")
print(f"   Input: '{test_text}'")

# ColBERT needs to tokenize first
test_colbert_embedding = colbert_model.encode(
    [test_text],  # ColBERT expects a list
    is_query=False  # False for documents, True for queries
)

print(f"   Output shape: {test_colbert_embedding[0].shape} (one vector per token)")
print(f"   This means: {test_colbert_embedding[0].shape[0]} tokens, each with {test_colbert_embedding[0].shape[1]} dimensions")

## 🤖 Load Models: Dense (Sentence-Transformers) vs ColBERT (PyLate)

In [ ]:
# Load ColBERT Model (Late Interaction)
from pylate import models

print("Loading ColBERT model...")
colbert_model = models.ColBERT(
    model_name_or_path='sentence-transformers/all-MiniLM-L6-v2',  # Using same base for fair comparison
    device='cpu'  # ColBERT also works better on CPU for M1
)

print(f"✅ ColBERT model loaded")
print(f"📏 ColBERT token embedding dimension: 384 per token")

# Test ColBERT embedding (returns multiple vectors, one per token)
test_tokens = colbert_model.encode(test_text, is_query=False)
print(f"🧪 Test ColBERT shape: {test_tokens.shape} (tokens x dimensions)")

In [ ]:
# Process all restaurant reviews
print("🔄 Creating embeddings for all restaurant reviews...")
print(f"📊 Processing {len(df)} reviews")

# Create dense embeddings (traditional RAG)
print("\n📦 Creating dense embeddings...")
dense_embeddings = []
for idx, row in df.iterrows():
    # Combine restaurant name and review for better context
    text = f"{row['restaurant']}: {row['review']}"
    embedding = dense_model.encode(text)
    dense_embeddings.append(embedding)
    print(f"  ✅ Dense embedding {idx+1}/{len(df)} - {row['restaurant']}")

print(f"✅ Created {len(dense_embeddings)} dense embeddings")
print(f"📏 Dense embedding shape: {dense_embeddings[0].shape}")

# Create ColBERT embeddings (late interaction)
print("\n🔍 Creating ColBERT embeddings...")
colbert_embeddings = []
for idx, row in df.iterrows():
    # Combine restaurant name and review for better context
    text = f"{row['restaurant']}: {row['review']}"
    # ColBERT expects a list and returns a list
    embedding = colbert_model.encode([text], is_query=False)
    colbert_embeddings.append(embedding[0])  # Get first (and only) embedding
    print(f"  ✅ ColBERT embedding {idx+1}/{len(df)} - {row['restaurant']}")

print(f"✅ Created {len(colbert_embeddings)} ColBERT embeddings")
print(f"📏 ColBERT embedding shape: {colbert_embeddings[0].shape} (tokens x dimensions)")

# Show the difference in storage requirements
import sys
dense_size = sum(sys.getsizeof(emb) for emb in dense_embeddings)
colbert_size = sum(sys.getsizeof(emb) for emb in colbert_embeddings)

print(f"\n💾 Storage comparison:")
print(f"   Dense embeddings: {dense_size:,} bytes ({dense_size/1024:.1f} KB)")
print(f"   ColBERT embeddings: {colbert_size:,} bytes ({colbert_size/1024:.1f} KB)")
print(f"   ColBERT is {colbert_size/dense_size:.1f}x larger (stores more information)")

# Prepare data for LanceDB storage
documents = []
for idx, row in df.iterrows():
    doc = {
        'id': int(row['id']),
        'restaurant': row['restaurant'],
        'review': row['review'],
        'reviewer': row['reviewer'],
        'rating': int(row['rating']),
        'text': f"{row['restaurant']}: {row['review']}",  # Combined text
        'dense_embedding': dense_embeddings[idx],
        'colbert_embedding': colbert_embeddings[idx]
    }
    documents.append(doc)

print(f"\n✅ Prepared {len(documents)} documents for LanceDB storage")

In [ ]:
# Create LanceDB tables for both embedding types
import pyarrow as pa

print("🗄️ Setting up LanceDB tables...")

# Create table for dense embeddings (traditional RAG)
print("\n📦 Creating dense embeddings table...")

# Schema for dense embeddings (single vector per document)
dense_schema = pa.schema([
    pa.field("id", pa.int64()),
    pa.field("restaurant", pa.string()),
    pa.field("review", pa.string()),
    pa.field("reviewer", pa.string()),
    pa.field("rating", pa.int64()),
    pa.field("text", pa.string()),
    pa.field("dense_embedding", pa.list_(pa.float32(), 384))  # Fixed size vector
])

# Prepare dense data
dense_data = []
for doc in documents:
    dense_row = {
        'id': doc['id'],
        'restaurant': doc['restaurant'],
        'review': doc['review'],
        'reviewer': doc['reviewer'],
        'rating': doc['rating'],
        'text': doc['text'],
        'dense_embedding': doc['dense_embedding'].tolist()  # Convert numpy to list
    }
    dense_data.append(dense_row)

# Create dense table
if "dense_reviews" in db.table_names():
    db.drop_table("dense_reviews")
    
dense_table = db.create_table("dense_reviews", dense_data, schema=dense_schema)
print(f"✅ Created dense_reviews table with {len(dense_table)} documents")

# Create table for ColBERT embeddings (multi-vector per document)
print("\n🔍 Creating ColBERT embeddings table...")

# For ColBERT, we need to flatten the multi-vector structure
# Each document will have multiple rows - one per token
colbert_data = []
for doc in documents:
    doc_id = doc['id']
    colbert_embedding = doc['colbert_embedding']  # Shape: (tokens, 384)
    
    # Create one row per token
    for token_idx, token_embedding in enumerate(colbert_embedding):
        colbert_row = {
            'doc_id': doc_id,
            'token_idx': token_idx,
            'restaurant': doc['restaurant'],
            'review': doc['review'],
            'reviewer': doc['reviewer'],
            'rating': doc['rating'],
            'text': doc['text'],
            'token_embedding': token_embedding.tolist()  # Convert numpy to list
        }
        colbert_data.append(colbert_row)

# Schema for ColBERT embeddings (one row per token)
colbert_schema = pa.schema([
    pa.field("doc_id", pa.int64()),
    pa.field("token_idx", pa.int64()),
    pa.field("restaurant", pa.string()),
    pa.field("review", pa.string()),
    pa.field("reviewer", pa.string()),
    pa.field("rating", pa.int64()),
    pa.field("text", pa.string()),
    pa.field("token_embedding", pa.list_(pa.float32(), 384))  # Fixed size vector per token
])

if "colbert_reviews" in db.table_names():
    db.drop_table("colbert_reviews")
    
colbert_table = db.create_table("colbert_reviews", colbert_data, schema=colbert_schema)
print(f"✅ Created colbert_reviews table with {len(colbert_table)} token embeddings")

# Summary
print(f"\n📊 LanceDB Storage Summary:")
print(f"   📦 Dense table: {len(dense_table)} documents")
print(f"   🔍 ColBERT table: {len(colbert_table)} token embeddings")
print(f"   📈 Average tokens per document: {len(colbert_table) / len(documents):.1f}")

# Display table info
print(f"\n🗂️ Tables in database: {db.table_names()}")
print(f"📏 Dense table schema: {[field.name for field in dense_table.schema]}")
print(f"📏 ColBERT table schema: {[field.name for field in colbert_table.schema]}")

In [ ]:
# Search functions for both approaches
def search_dense(query, top_k=3):
    """Search using dense embeddings (traditional RAG)"""
    # Encode query with dense model
    query_embedding = dense_model.encode(query)
    
    # Search LanceDB dense table
    results = dense_table.search(query_embedding).limit(top_k).to_pandas()
    
    return results

def search_colbert(query, top_k=3):
    """Search using ColBERT late interaction"""
    # Encode query with ColBERT model
    query_embeddings = colbert_model.encode([query], is_query=True)[0]  # Shape: (query_tokens, 384)
    
    # For ColBERT, we need to implement MaxSim operation
    # This is a simplified version - in practice, you'd use optimized vector search
    doc_scores = {}
    
    # Get all token embeddings from ColBERT table
    all_tokens = colbert_table.to_pandas()
    
    # Group by document and calculate MaxSim scores
    for doc_id in all_tokens['doc_id'].unique():
        doc_tokens = all_tokens[all_tokens['doc_id'] == doc_id]
        
        # Get token embeddings for this document
        doc_embeddings = np.array([token['token_embedding'] for _, token in doc_tokens.iterrows()])
        
        # Calculate MaxSim: for each query token, find max similarity with any doc token
        max_sims = []
        for query_token in query_embeddings:
            # Cosine similarity between query token and all doc tokens
            similarities = np.dot(doc_embeddings, query_token) / (
                np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(query_token)
            )\n            max_sims.append(np.max(similarities))\n        \n        # Final score is sum of all max similarities\n        doc_scores[doc_id] = np.sum(max_sims)\n    \n    # Sort by score and get top_k\n    sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]\n    \n    # Get document details\n    results = []\n    for doc_id, score in sorted_docs:\n        doc_info = all_tokens[all_tokens['doc_id'] == doc_id].iloc[0]\n        result = {\n            'id': doc_info['doc_id'],\n            'restaurant': doc_info['restaurant'],\n            'review': doc_info['review'],\n            'reviewer': doc_info['reviewer'],\n            'rating': doc_info['rating'],\n            'score': score\n        }\n        results.append(result)\n    \n    return pd.DataFrame(results)\n\nprint(\"✅ Search functions ready!\")\nprint(\"📦 Dense search: Single vector cosine similarity\")\nprint(\"🔍 ColBERT search: MaxSim late interaction\")

In [ ]:
# Let's test some complex queries that show the difference!

def compare_search_results(query, description):
    \"\"\"Compare dense vs ColBERT search results side by side\"\"\"\n    print(f\"\\n{'='*80}\")\n    print(f\"🔍 QUERY: {query}\")\n    print(f\"💡 {description}\")\n    print(f\"{'='*80}\")\n    \n    # Get results from both approaches\n    print(\"\\n📦 DENSE RETRIEVAL (Traditional RAG):\")\n    print(\"-\" * 50)\n    dense_results = search_dense(query, top_k=3)\n    for idx, row in dense_results.iterrows():\n        score = row.get('_distance', 'N/A')  # LanceDB returns distance, lower is better\n        print(f\"\\n#{row['id']} - {row['restaurant']} | ⭐{row['rating']} | Score: {score}\")\n        print(f\"Review: {row['review'][:150]}...\")\n    \n    print(\"\\n\\n🔍 COLBERT (Late Interaction):\")\n    print(\"-\" * 50)\n    colbert_results = search_colbert(query, top_k=3)\n    for idx, row in colbert_results.iterrows():\n        print(f\"\\n#{row['id']} - {row['restaurant']} | ⭐{row['rating']} | Score: {row['score']:.3f}\")\n        print(f\"Review: {row['review'][:150]}...\")\n    \n    return dense_results, colbert_results\n\nprint(\"✅ Comparison function ready!\")

In [ ]:
# Test Case 1: Multi-constraint query\nquery1 = \"Italian budget-friendly outdoor seating\"\ndescription1 = \"Testing multi-constraint matching: Italian + affordable + outdoor\"\ndense1, colbert1 = compare_search_results(query1, description1)

In [ ]:
# Test Case 2: Contradictory concepts\nquery2 = \"expensive but worth it fine dining\"\ndescription2 = \"Testing contradictory concepts: expensive + worth it (value)\"\ndense2, colbert2 = compare_search_results(query2, description2)

In [ ]:
# Test Case 3: Work-friendly cafe\nquery3 = \"laptop work wifi quiet productive\"\ndescription3 = \"Testing work environment matching: workspace + quiet + wifi\"\ndense3, colbert3 = compare_search_results(query3, description3)

## 🎨 Token-Level Visualization

See exactly HOW ColBERT matches individual tokens!

In [ ]:
# Search functions for both approaches
def search_dense(query, top_k=3):
    """Search using dense embeddings (traditional RAG)"""
    # Encode query with dense model
    query_embedding = dense_model.encode(query)
    
    # Search LanceDB dense table
    results = dense_table.search(query_embedding).limit(top_k).to_pandas()
    
    return results

def search_colbert(query, top_k=3):
    """Search using ColBERT late interaction"""
    # Encode query with ColBERT model
    query_embeddings = colbert_model.encode([query], is_query=True)[0]  # Shape: (query_tokens, 384)
    
    # For ColBERT, we need to implement MaxSim operation
    # This is a simplified version - in practice, you'd use optimized vector search
    doc_scores = {}
    
    # Get all token embeddings from ColBERT table
    all_tokens = colbert_table.to_pandas()
    
    # Group by document and calculate MaxSim scores
    for doc_id in all_tokens['doc_id'].unique():
        doc_tokens = all_tokens[all_tokens['doc_id'] == doc_id]
        
        # Get token embeddings for this document
        doc_embeddings = np.array([token['token_embedding'] for _, token in doc_tokens.iterrows()])
        
        # Calculate MaxSim: for each query token, find max similarity with any doc token
        max_sims = []
        for query_token in query_embeddings:
            # Cosine similarity between query token and all doc tokens
            similarities = np.dot(doc_embeddings, query_token) / (
                np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(query_token)
            )
            max_sims.append(np.max(similarities))
        
        # Final score is sum of all max similarities
        doc_scores[doc_id] = np.sum(max_sims)
    
    # Sort by score and get top_k
    sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    
    # Get document details
    results = []
    for doc_id, score in sorted_docs:
        doc_info = all_tokens[all_tokens['doc_id'] == doc_id].iloc[0]
        result = {
            'id': doc_info['doc_id'],
            'restaurant': doc_info['restaurant'],
            'review': doc_info['review'],
            'reviewer': doc_info['reviewer'],
            'rating': doc_info['rating'],
            'score': score
        }
        results.append(result)
    
    return pd.DataFrame(results)

print("✅ Search functions ready!")
print("📦 Dense search: Single vector cosine similarity")
print("🔍 ColBERT search: MaxSim late interaction")

In [ ]:
# Visualize token matching for our best results\nprint(\"🎨 Let's see HOW ColBERT found the best matches!\")\n\n# Visualize the Italian query matching\nprint(\"\\n\" + \"=\"*80)\nprint(\"CASE STUDY: Italian budget-friendly outdoor seating\")\nprint(\"=\"*80)\n\n# Get the top ColBERT result for Italian query\nif len(colbert1) > 0:\n    best_colbert_doc = colbert1.iloc[0]['id']\n    visualize_colbert_matching(\"Italian budget-friendly outdoor\", best_colbert_doc)\nelse:\n    print(\"No ColBERT results found for Italian query\")\n\n# Show why this works better than dense\nprint(\"\\n💡 WHY COLBERT WINS:\")\nprint(\"   ✅ 'Italian' token matches 'Italian' in text exactly\")\nprint(\"   ✅ 'budget' token matches '$28' pricing context\")\nprint(\"   ✅ 'outdoor' token matches 'outdoor patio' exactly\")\nprint(\"   ❌ Dense embedding compresses all this into single point!\")

## 🎯 Demo Summary: Why ColBERT Beats Dense Retrieval

The key insights from our restaurant reviews demo:

In [ ]:
# Final comparison and insights\nprint(\"🎯 DEMO INSIGHTS: Why ColBERT Beats Dense Retrieval\")\nprint(\"=\"*80)\n\nprint(\"\\n📊 STORAGE COMPARISON:\")\nprint(f\"   📦 Dense: {len(dense_embeddings)} vectors × 384 dims = {len(dense_embeddings) * 384:,} numbers\")\nprint(f\"   🔍 ColBERT: ~{len(colbert_table)} tokens × 384 dims = {len(colbert_table) * 384:,} numbers\")\nprint(f\"   💾 ColBERT uses {(len(colbert_table) * 384) / (len(dense_embeddings) * 384):.1f}x more storage\")\nprint(\"   💡 But preserves much more information!\")\n\nprint(\"\\n🔍 SEARCH QUALITY:\")\nprint(\"   📦 Dense Retrieval:\")\nprint(\"      ❌ Single vector = compressed information\")\nprint(\"      ❌ Struggles with multi-constraint queries\")\nprint(\"      ❌ Can't handle contradictory concepts well\")\nprint(\"      ✅ Fast and simple\")\nprint(\"      ✅ Lower storage requirements\")\n\nprint(\"\\n   🔍 ColBERT Late Interaction:\")\nprint(\"      ✅ Token-level precision matching\")\nprint(\"      ✅ Excellent multi-constraint queries\")\nprint(\"      ✅ Handles contradictory concepts\")\nprint(\"      ✅ Interpretable token matches\")\nprint(\"      ❌ Higher storage requirements\")\nprint(\"      ❌ More complex search process\")\n\nprint(\"\\n🎯 KEY TAKEAWAY FOR AI TINKERERS:\")\nprint(\"   Late interaction models like ColBERT represent a fundamental\")\nprint(\"   shift from 'lossy compression' to 'lossless precision' in RAG.\")\nprint(\"   Perfect for complex, multi-faceted queries where accuracy matters!\")\n\nprint(\"\\n🚀 NEXT STEPS:\")\nprint(\"   1. Multiple Representations: Use different embedding models\")\nprint(\"   2. Reasoning Retrievers: Add query decomposition & analysis\")\nprint(\"   3. Hybrid Approaches: Combine dense + ColBERT strategically\")